# SAR Pattern Detection — FATF Typology Rules

This notebook demonstrates rules-based detection of Suspicious Activity Report (SAR)
patterns using the `aml-analytics` toolkit.

Four FATF/FinCEN typologies are implemented:
- **Structuring** — breaking amounts below the $10,000 CTR threshold (FinCEN FIN-2014-A005)
- **Smurfing** — multiple senders depositing just-below-threshold amounts to one receiver
- **Round-tripping** — funds sent and returned between the same accounts
- **Cash-intensive** — accounts with unusually high proportions of cash transactions

Each rule cites the specific FinCEN advisory or FATF report it implements.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import matplotlib.pyplot as plt

from aml.synthetic import generate_transactions
from aml.patterns import (
    detect_structuring_pattern,
    detect_smurfing_pattern,
    detect_round_tripping,
    detect_cash_intensive,
    run_all,
)

print("Modules loaded")

## 1. Generate synthetic transaction data

In [ ]:
df = generate_transactions(n=1000, seed=42)
print(f"Transactions : {len(df):,}")
print(f"Accounts     : {df['sender_id'].nunique()}")
print(f"\nLabel breakdown:")
print(df['label'].value_counts().to_string())

## 2. Structuring detection
Accounts making multiple cash transactions just below the $10,000 CTR threshold within 48 hours.
Reference: FinCEN Advisory FIN-2014-A005

In [ ]:
structuring = detect_structuring_pattern(df, threshold=9_000, window_hours=48)
print(f"Structuring alerts: {len(structuring)}")
if len(structuring) > 0:
    print(structuring[['sender_id','transaction_count','total_amount',
                        'avg_amount','risk_score','fincen_reference']].head(10))

## 3. Smurfing detection
Multiple senders each depositing just-below-threshold amounts to the same receiver.
Reference: FATF Glossary — Smurfing (2012)

In [ ]:
smurfing = detect_smurfing_pattern(df, threshold=9_000, window_hours=24, min_senders=2)
print(f"Smurfing alerts: {len(smurfing)}")
if len(smurfing) > 0:
    print(smurfing[['receiver_id','unique_senders','transaction_count',
                    'total_amount','risk_score','fincen_reference']].head(10))

## 4. Round-tripping detection
Funds sent from Account A to Account B and returned within 72 hours.
Reference: FinCEN Advisory FIN-2012-A001

In [ ]:
round_trips = detect_round_tripping(df, window_hours=72, amount_tolerance=0.05)
print(f"Round-tripping alerts: {len(round_trips)}")
if len(round_trips) > 0:
    print(round_trips[['account_a','account_b','outgoing_amount',
                        'return_amount','hours_elapsed','fincen_reference']].head(10))

## 5. Cash-intensive business risk
Accounts where more than 70% of transactions are cash-based.
Reference: FinCEN Advisory FIN-2014-A005

In [ ]:
cash = detect_cash_intensive(df, cash_threshold=0.5, min_transactions=3)
print(f"Cash-intensive alerts: {len(cash)}")
if len(cash) > 0:
    print(cash[['account_id','total_transactions','cash_transactions',
                'cash_ratio','total_cash_amount','fincen_reference']].head(10))

## 6. Run all typologies at once

In [ ]:
results = run_all(df)
print("Alert summary across all typologies:\n")
print(results['summary'].to_string(index=False))

In [ ]:
# Visualize alert counts by typology
summary = results['summary']

plt.figure(figsize=(9, 5))
colors = ['steelblue', 'darkorange', 'seagreen', 'crimson']
bars = plt.bar(summary['typology'], summary['alert_count'], color=colors, edgecolor='white')
plt.bar_label(bars, padding=3, fontsize=11)
plt.xlabel('Typology', fontsize=11)
plt.ylabel('Alert Count', fontsize=11)
plt.title('SAR Pattern Alerts by FATF Typology', fontsize=13)
plt.tight_layout()
plt.savefig('sar_alerts.png', dpi=150, bbox_inches='tight')
plt.show()

## Summary

| Typology | FinCEN Reference | Alerts |
|---|---|---|
| Structuring | FIN-2014-A005 | See above |
| Smurfing | FATF Glossary 2012 | See above |
| Round-tripping | FIN-2012-A001 | See above |
| Cash-intensive | FIN-2014-A005 | See above |

Rules-based detection provides explainable, auditable alerts — each alert can be
directly tied to a regulatory reference, which is essential for SAR narrative writing
and examiner defensibility.

Toolkit: [github.com/Bhavesh0205/aml-analytics](https://github.com/Bhavesh0205/aml-analytics)